<a href="https://colab.research.google.com/github/ludan369/30-Days-Of-Python/blob/master/nb/Qwen2.5_(3B)-GRPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://docs.unsloth.ai/get-started/installing-+-updating).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News

**Read our [blog post](https://unsloth.ai/blog/r1-reasoning) for guidance on how to train reasoning models.**

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [1]:
%%capture
# Skip restarting message in Colab
import sys; modules = list(sys.modules.keys())
for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None

!pip install unsloth vllm
!pip install --upgrade pillow
# If you are running this notebook on local, you need to install `diffusers` too
# !pip install diffusers
# Temporarily install a specific TRL nightly version
!pip install git+https://github.com/huggingface/trl.git@e95f9fb74a3c3647b86f251b7e230ec51c64b72b

### Unsloth

Use `PatchFastRL` before all functions to patch GRPO and other RL algorithms!

In [2]:
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)

Unsloth: Patching Xformers to fix some performance issues.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 02-12 04:44:55 __init__.py:190] Automatically detected platform cuda.


Load up `Qwen 2.5 3B Instruct`, and set parameters

In [3]:
from unsloth import is_bfloat16_supported
import torch
max_seq_length = 1024 # Can increase for longer reasoning traces
lora_rank = 64 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.5, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ], # Remove QKVO if out of memory
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth", # Enable long context finetuning
    random_state = 3407,
)

==((====))==  Unsloth 2025.2.5: Fast Qwen2 patching. Transformers: 4.48.2.
   \\   /|    GPU: Tesla T4. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit with actual GPU utilization = 49.66%
Unsloth: Your GPU has CUDA compute capability 7.5 with VRAM = 14.74 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 1024. Num Sequences = 192.
Unsloth: vLLM's KV Cache can use up to 4.9 GB. Also swap space = 2 GB.
WARNING 02-12 04:45:12 config.py:2386] Casting torch.bfloat16 to torch.float16.
INFO 02-12 04:45:25 config.py:542] This model supports multiple tasks: {'reward', 'classify', 'embed', 'generate', 'score'}. Def

tokenizer_config.json:   0%|          | 0.00/7.36k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Unsloth 2025.2.5 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


### Data Prep
<a name="Data"></a>

We directly leverage [@willccbb](https://gist.github.com/willccbb/4676755236bb08cab5f4e54a0475d6fb) for data prep and all reward functions. You are free to create your own!

In [4]:
import re
from datasets import load_dataset, Dataset

# 修改系统提示为中文医疗场景
SYSTEM_PROMPT = """
你是一个专业的医疗助手。请按照以下格式分析病例并给出诊断：
<reasoning>
1. 分析症状
2. 考虑可能原因
3. 给出诊断建议
</reasoning>
<answer>
最终诊断或建议
</answer>
"""

XML_COT_FORMAT = """\
<reasoning>
{reasoning}
</reasoning>
<answer>
{answer}
</answer>
"""


def extract_xml_answer(text: str) -> str:
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

def get_medical_dataset(split = "train") -> Dataset:
    """
    加载医疗数据集
    字段：Question（问题）、Complex_CoT（推理过程）、Response（答案）
    """
    # 加载数据集
    data = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "zh")[split]

    def process_example(x):
        # 构建prompt
        prompt = [
            {'role': 'system', 'content': SYSTEM_PROMPT}
        ]

        # 可以取消注释以添加few-shot示例
        """
        if random.random() < 0.2:  # 20%概率添加示例
            prompt.extend([
                {'role': 'user', 'content': '示例问题'},
                {'role': 'assistant', 'content': XML_COT_FORMAT.format(
                    reasoning='示例推理过程',
                    answer='示例答案'
                )}
            ])
        """

        # 添加当前问题
        prompt.append({'role': 'user', 'content': x['Question']})

        return {
            'prompt': prompt,
            'answer': x['Response'],
            'cot': x['Complex_CoT']  # 保存原始推理过程用于评估
        }

    return data.map(process_example)


def medical_correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    q = prompts[0][-1]['content']
    extracted_responses = [extract_xml_answer(r) for r in responses]

    print('-'*20,
          f"\n问题:\n{q}",
          f"\n标准答案:\n{answer[0]}",
          f"\n模型回答:\n{responses[0]}",
          f"\n提取的答案:\n{extracted_responses[0]}")

    # 简单的答案匹配（可以根据需要改进）
    def calculate_medical_similarity(pred, true):
        # 这里可以添加更复杂的中文医疗文本相似度计算
        # 例如：关键医疗术语的匹配度、诊断要点的覆盖率等
        if pred.strip() == true.strip():
            return 2.0
        # 部分匹配给予部分分数
        elif any(key in pred for key in true.split()):
            return 1.0
        return 0.0

    return [calculate_medical_similarity(r, a) for r, a in zip(extracted_responses, answer)]

def reasoning_quality_reward_func(completions, cot, **kwargs) -> list[float]:
    """评估推理过程的质量"""
    responses = [completion[0]['content'] for completion in completions]

    def extract_reasoning(text):
        try:
            reasoning = text.split("<reasoning>")[-1]
            reasoning = reasoning.split("</reasoning>")[0]
            return reasoning.strip()
        except:
            return ""

    def evaluate_reasoning_quality(pred_reasoning, true_reasoning):
        # 评估推理质量
        # 1. 分析关键步骤的覆盖
        true_steps = set(true_reasoning.split('\n'))
        pred_steps = set(pred_reasoning.split('\n'))
        step_coverage = len(true_steps.intersection(pred_steps)) / len(true_steps)

        # 2. 检查医疗术语的使用
        medical_terms = ['症状', '诊断', '建议', '治疗', '检查']  # 可以扩充
        term_usage = sum(1 for term in medical_terms if term in pred_reasoning) / len(medical_terms)

        # 3. 计算综合得分
        score = (step_coverage * 0.7 + term_usage * 0.3)
        return min(1.0, score)

    extracted_reasonings = [extract_reasoning(r) for r in responses]
    return [evaluate_reasoning_quality(r, c) for r, c in zip(extracted_reasonings, cot)]

# 格式相关的奖励函数
def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """严格的格式检查"""
    # 中文格式的正则表达式模式
    pattern = r"^<reasoning>\n(?:[\s\S]*?)\n</reasoning>\n<answer>\n(?:[\s\S]*?)\n</answer>\n$"

    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]

    def check_format_quality(response):
        # 检查是否包含基本结构
        has_basic_structure = bool(re.match(pattern, response))
        if not has_basic_structure:
            return 0.0

        # 检查推理部分的结构
        reasoning = re.findall(r"<reasoning>([\s\S]*?)</reasoning>", response)
        if not reasoning:
            return 0.0

        reasoning_content = reasoning[0].strip()
        # 检查推理是否包含数字编号或步骤说明
        has_steps = bool(re.search(r"[1-9][.、]|步骤[1-9]|第[一二三四五六七八九]步", reasoning_content))

        # 检查答案部分
        answer = re.findall(r"<answer>([\s\S]*?)</answer>", response)
        has_clear_answer = bool(answer and answer[0].strip())

        # 计算总分
        score = 0.5  # 基本结构正确
        if has_steps:
            score += 0.25  # 有清晰的步骤
        if has_clear_answer:
            score += 0.25  # 有清晰的答案

        return score

    return [check_format_quality(r) for r in responses]

def soft_format_reward_func(completions, **kwargs) -> list[float]:
    """相对宽松的格式检查"""
    pattern = r"<reasoning>[\s\S]*?</reasoning>\s*<answer>[\s\S]*?</answer>"
    responses = [completion[0]["content"] for completion in completions]

    def evaluate_soft_format(response):
        # 基本格式检查
        if not re.search(pattern, response):
            return 0.0

        score = 0.0

        # 1. 检查标签完整性 (0.2分)
        tags = ["<reasoning>", "</reasoning>", "<answer>", "</answer>"]
        score += 0.2 * sum(tag in response for tag in tags) / len(tags)

        # 2. 检查内容结构 (0.2分)
        try:
            reasoning = response.split("<reasoning>")[1].split("</reasoning>")[0].strip()
            answer = response.split("<answer>")[1].split("</answer>")[0].strip()

            if reasoning and answer:
                score += 0.2
        except:
            pass

        # 3. 检查推理格式 (0.3分)
        if reasoning:
            # 检查是否有序号或关键词
            has_numbers = bool(re.search(r"[1-9][.、]|步骤[1-9]|第[一二三四五六七八九]步", reasoning))
            has_keywords = bool(re.search(r"首先|其次|然后|最后|综上|因此", reasoning))

            if has_numbers:
                score += 0.15
            if has_keywords:
                score += 0.15

        # 4. 检查医疗相关格式 (0.3分)
        medical_patterns = [
            r"症状[：:](.*?)[。\n]",
            r"诊断[：:](.*?)[。\n]",
            r"建议[：:](.*?)[。\n]",
            r"治疗[：:](.*?)[。\n]"
        ]

        score += 0.3 * sum(bool(re.search(p, response)) for p in medical_patterns) / len(medical_patterns)

        return min(1.0, score)

    return [evaluate_soft_format(r) for r in responses]

def count_xml_medical(text) -> float:
    """检查XML结构的完整性，针对医疗场景优化"""
    score = 0.0

    # 1. 基本标签检查 (0.4分)
    if text.count("<reasoning>\n") == 1:
        score += 0.1
    if text.count("\n</reasoning>\n") == 1:
        score += 0.1
    if text.count("\n<answer>\n") == 1:
        score += 0.1
    if text.count("\n</answer>") == 1:
        score += 0.1

    # 2. 内容结构检查 (0.3分)
    try:
        reasoning = text.split("<reasoning>")[1].split("</reasoning>")[0].strip()
        answer = text.split("<answer>")[1].split("</answer>")[0].strip()

        # 检查推理部分是否有步骤
        if re.search(r"[1-9][.、]|步骤[1-9]|第[一二三四五六七八九]步", reasoning):
            score += 0.15

        # 检查答案部分是否简洁明确
        if len(answer.split('\n')) <= 3 and answer.strip():
            score += 0.15
    except:
        pass

    # 3. 医疗专业性检查 (0.3分)
    medical_terms = [
        r"症状",
        r"诊断",
        r"建议",
        r"治疗",
        r"检查",
        r"病因",
        r"并发症"
    ]

    term_score = sum(bool(re.search(term, text)) for term in medical_terms) / len(medical_terms)
    score += term_score * 0.3

    return min(1.0, score)

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    """优化后的XML结构评分"""
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml_medical(c) for c in contents]

# 使用示例
dataset = get_medical_dataset()

README.md:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

medical_o1_sft_Chinese.json:   0%|          | 0.00/64.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/24772 [00:00<?, ? examples/s]

Map:   0%|          | 0/24772 [00:00<?, ? examples/s]

<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [5]:
from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    use_vllm = True, # use vLLM for fast inference!
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    bf16 = is_bfloat16_supported(),
    fp16 = not is_bfloat16_supported(),
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1, # Increase to 4 for smoother training
    num_generations = 8, # Decrease if out of memory
    max_prompt_length = 256,
    max_completion_length = 200,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 250,
    save_steps = 250,
    max_grad_norm = 0.1,
    report_to = "none", # Can use Weights & Biases
    output_dir = "outputs",
)

torch.distributed process group is initialized, but parallel_mode != ParallelMode.DISTRIBUTED. In order to use Torch DDP, launch your script with `python -m torch.distributed.launch


And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |


In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        medical_correctness_reward_func,
        reasoning_quality_reward_func,
        strict_format_reward_func,
        soft_format_reward_func,
        xmlcount_reward_func
    ],
    args = training_args,
    train_dataset = dataset,
)
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 24,772 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 1 | Gradient Accumulation steps = 1
\        /    Total batch size = 1 | Total steps = 250
 "-____-"     Number of trainable parameters = 119,734,272


-------------------- 
问题:
Given a cascaded amplifier with three stages having power gains of G_1 = 15, G_2 = 10, and G_3 = 5, and effective input noise temperatures of T_e1 = 1350K, T_e2 = 1700K, and T_e3 = 2500K, calculate the overall effective noise temperature of the amplifier. Additionally, determine the optimal ordering of these stages to achieve the best noise performance. 
标准答案:
To calculate the overall effective noise temperature of a cascaded amplifier, we use the Friis formula for noise in cascaded systems. The formula for the overall noise temperature, \( T_{e\_total} \), is given by:

\[ T_{e\_total} = T_{e1} + \frac{T_{e2}}{G_1} + \frac{T_{e3}}{G_1 \times G_2} \]

Given the parameters:

- \( G_1 = 15 \)
- \( G_2 = 10 \)
- \( G_3 = 5 \)
- \( T_{e1} = 1350 \text{ K} \)
- \( T_{e2} = 1700 \text{ K} \)
- \( T_{e3} = 2500 \text{ K} \)

First, calculate the contribution from the second and third stages: 

1. Contribution from the second stage:
   \[ \frac{T_{e2}}{G_1} = \frac{17

Step,Training Loss,reward,reward_std,completion_length,kl
1,0.000000,1.571039,0.147811,199.750000,0.000021
2,0.000000,1.075357,0.071721,200.000000,0.000013
3,0.000000,0.976964,0.614020,198.000000,0.000011
4,0.000000,1.072917,0.060381,137.750000,0.000018
5,-0.000000,1.499206,0.036365,200.000000,0.000011
6,0.000000,1.016667,0.634789,200.000000,0.000010
7,0.000000,0.811012,0.572911,195.250000,0.000014
8,0.000000,1.482857,0.072731,200.000000,0.000012
9,0.000000,1.350844,0.109244,200.000000,0.000013
10,0.000000,1.091429,0.354335,200.000000,0.000018


-------------------- 
问题:
Given that the separation of the headlights on a car is 150 cm and the car is 30 meters away from the observer, calculate the separation of the images of the headlights on the retina. 
标准答案:
To calculate the separation of the images of the car's headlights on the retina, we can determine the angular separation of the headlights and then use the properties of the eye to project this onto the retina.

1. **Determine the Angular Separation:**
   - The distance between the headlights is 150 cm.
   - The car is 30 meters (or 3000 cm) away from the observer.
   - The angular separation, θ, can be approximated for small angles as θ = separation / distance = 150 cm / 3000 cm = 0.05 radians.

2. **Translate Angular Separation to Retinal Image Separation:**
   - The average focal length of an eye (distance from lens to retina) is about 2 cm.
   - The separation on the retina, s, is given by s = θ × focal length of the eye.
   - So, s = 0.05 radians × 2 cm = 0.1 cm.

Hen

<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "user", "content" : "How many r's are in strawberry?"},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.71it/s, est. speed input: 63.38 toks/s, output: 25.69 toks/s]


'There are 2 r\'s in the word "strawberry."'

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [ ]:
model.save_lora("grpo_saved_lora")

Now we load the LoRA and test:

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "system", "content" : SYSTEM_PROMPT},
    {"role" : "user", "content" : "How many r's are in strawberry?"},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.06s/it, est. speed input: 14.05 toks/s, output: 29.09 toks/s]


'<reasoning>\nTo find out how many times the letter \'r\' appears in the word "strawberry", we can go through the word character by character and count each occurrence of \'r\'. In "strawberry", the letter \'r\' appears 3 times: once in the beginning, once in the middle, and once at the end of the word.\n</reasoning>\n<answer>\n3\n</answer>'

Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/drive/1WZDi7APtQ9VsvOrQSSC5DDtxq159j8iZ?usp=sharing)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "",
    )

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp or a UI based system like Jan or Open WebUI. You can install Jan [here](https://github.com/janhq/jan) and Open WebUI [here](https://github.com/open-webui/open-webui)

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Llama 3.2 Conversational notebook. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(1B_and_3B)-Conversational.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
